In [1]:
import geopandas as gpd
import geemap
import ee

import os
from glob import glob

In [2]:
# List available cities in mnt/
cities = [d for d in os.listdir('mnt') if os.path.isdir(f'mnt/{d}') and d.startswith('20')]
print("Available cities:")

for i, city in enumerate(sorted(cities), 1):
    print(f"{i}. {city}")


Available cities:
1. 2025-02-tunisia-tunis
2. 2025-10-indonesia-sofifi
3. 2025-10-senegal-dakar
4. 2025-10-senegal-dakar-old
5. 2025-10-senegal-diourbel
6. 2025-10-senegal-matam
7. 2025-10-senegal-matam-old
8. 2025-10-senegal-tambacounda
9. 2025-11-mauritania-nouakchott
10. 2025-11-mauritania-nouakchott-old
11. 2025-12-senegal-dagana
12. 2025-12-senegal-dakar
13. 2025-12-senegal-diourbel
14. 2025-12-senegal-matam
15. 2025-12-senegal-mbour
16. 2025-12-senegal-tambacounda


In [3]:
# Select city
city_dir = sorted(cities)[8] # Change this to your city

cityname  = city_dir.split('-')[-1] 
data_dir = os.path.join('mnt', city_dir, '02-process-output', 'spatial')
aoi_dir = os.path.join('mnt', city_dir, '01-user-input', 'AOI')
aoi_file = glob(aoi_dir + "/*.shp")[0]

print(f"\n{cityname} - selected: {city_dir}. data_dir: {data_dir}")
print(aoi_file)




nouakchott - selected: 2025-11-mauritania-nouakchott. data_dir: mnt/2025-11-mauritania-nouakchott/02-process-output/spatial
mnt/2025-11-mauritania-nouakchott/01-user-input/AOI/nouakchott.shp


In [4]:
# Initialize Earth Engine
ee.Authenticate()
ee.Initialize(project = "acr-dev-450519")

In [5]:
# ============================================
# 1. LOAD AOI FROM LOCAL SHAPEFILE
# ============================================
aoi_gdf = gpd.read_file(aoi_file)
aoi = geemap.gdf_to_ee(aoi_gdf)

# Define time range
start_year = 2016
end_year = 2025

In [ ]:
# import numpy as np
# ghs_builts = ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S')\
#     .map(lambda img: img.clip(aoi)) 

# for year in np.arange(1975, 2035, 5):
#     print(year)

#     g = ghs_builts.filter(ee.Filter.eq('system:index', str(year)))

#     ee.batch.Export.image.toDrive(
#         image=g.first(),
#         description=f'{cityname}_ghs_built_{year}',
#         folder='GEE_Exports',
#         fileNamePrefix=f'{cityname}_ghs_built_{year}',
#         region=aoi.geometry(),
#         # crs='EPSG:4326',
#         # scale=100,
#         maxPixels=1e13
#     ).start()

1975
1980
1985
1990
1995
2000
2005
2010
2015
2020
2025
2030


In [6]:
aqueduct = ee.FeatureCollection("WRI/Aqueduct_Water_Risk/V4/baseline_annual")
aqueduct_aoi = aqueduct.filterBounds(aoi)

# Export as vector (GeoJSON or SHP)
ee.batch.Export.table.toDrive(
collection=aqueduct_aoi,
description=f'{cityname}_water_risk',
folder='GEE_Exports',
fileNamePrefix=f'{cityname}_water_risk',
fileFormat='GeoJSON'  # or 'SHP' for shapefile
).start()

print(f'Started export: {cityname}_water_risk')

Started export: nouakchott_water_risk


In [8]:
criteria = ee.Filter.And([
    ee.Filter.calendarRange(start_year, end_year, 'year'), 
    ee.Filter.bounds(aoi)  
])

In [9]:
def get_yearly_s2(year):
    year = ee.Number(year)
    
    return (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(aoi)
        .filter(ee.Filter.calendarRange(year, year, 'year'))
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
        .select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12'])
        .median()
        .clip(aoi)
        .set('year', year))

def get_yearly_s1(year):
    year = ee.Number(year)
    
    return (ee.ImageCollection('COPERNICUS/S1_GRD')
        .filterBounds(aoi)
        .filter(ee.Filter.calendarRange(year, year, 'year'))
        .filter(ee.Filter.eq('instrumentMode', 'IW'))
        .select(['VV', 'VH'])
        .mean()
        .clip(aoi)
        .set('year', year))

def get_yearly_viirs(year):
    year = ee.Number(year)
    return (ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
        .filterBounds(aoi)
        .filter(ee.Filter.calendarRange(year, year, 'year'))
        .select('avg_rad')
        .min()
        .clip(aoi)
        .set('year', year))





In [10]:
years = ee.List.sequence(2016, 2025)

s2 = ee.ImageCollection.fromImages(years.map(get_yearly_s2))
s1 = ee.ImageCollection.fromImages(years.map(get_yearly_s1))
nl = ee.ImageCollection.fromImages(years.map(get_yearly_viirs))




In [11]:
# ============================================
gdp_hdi = ee.Image("projects/sat-io/open-datasets/GRIDDED_HDI_GDP/total_gdp_perCapita_1990_2020_30arcsec").clip(aoi)
gdp_hdi

In [12]:
Map = geemap.Map()
Map.centerObject(aoi, 10)
Map.addLayer(aoi, {'color': 'red'}, 'AOI')
Map.addLayer(gdp_hdi.select('PPP_2020'), {'min': 0, 'max': 5000000, 'palette': ['blue', 'yellow', 'red']}, 'GDP PPP')

    
# Map.addLayer(s2.first(), 
#             {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, "Sentinel-2 RGB", False)

Map.addLayer(nl.first(), 
            {'min': 0, 'max': 50, 'palette': ['black', 'yellow', 'white']}, "Nightlight", False)

Map


Map(center=[18.07511879541702, -15.96759430361697], controls=(WidgetControl(options=['position', 'transparent_…

In [13]:
# Export each year as separate TIF

for year in range(2016, 2025):
    img_s1 = s1.filter(ee.Filter.eq('year', year)).first()
    img_s2 = s2.filter(ee.Filter.eq('year', year)).first()
    img_nl = nl.filter(ee.Filter.eq('year', year)).first()

    ee.batch.Export.image.toDrive(
        image=img_s2,
        description=f'{cityname}_s2_{year}',
        folder='GEE_Exports',
        fileNamePrefix=f's2_{year}',
        region=aoi.geometry(),
        scale=10,
        maxPixels=1e13
    ).start()
    
    ee.batch.Export.image.toDrive(
        image=img_s1,
        description=f'{cityname}_s1_{year}',
        folder='GEE_Exports',
        fileNamePrefix=f'{cityname}_s1_{year}',
        region=aoi.geometry(),
        scale=10,
        maxPixels=1e13
    ).start()
    
    ee.batch.Export.image.toDrive(
        image=img_nl,
        description=f'{cityname}_nightlight_{year}',
        folder='GEE_Exports',
        fileNamePrefix=f'{cityname}_nightlight_{year}',
        region=aoi.geometry(),
        scale=500,
        maxPixels=1e13
    ).start()



In [ ]:
import numpy as np
ghs_builts = ee.ImageCollection('JRC/GHSL/P2023A/GHS_BUILT_S')\
    .map(lambda img: img.clip(aoi)) 

for year in np.arange(1975, 2035, 5):
    print(year)

    g = ghs_builts.filter(ee.Filter.eq('system:index', str(year)))

    ee.batch.Export.image.toDrive(
        image=g.first(),
        description=f'{cityname}_ghs_built_{year}',
        folder='GEE_Exports',
        fileNamePrefix=f'{cityname}_ghs_built_{year}',
        region=aoi.geometry(),
        # crs='EPSG:4326',
        # scale=100,
        maxPixels=1e13
    ).start()

1975
1980
1985
1990
1995
2000
2005
2010
2015
2020
2025
2030
